# 09 - Comparacao de Modelos e Escolha Final

Este notebook implementa as tarefas 20, 21 e 22 da ordem recomendada:

1. Consolidar metricas.
2. Comparar desempenho e eficiencia.
3. Escolher o modelo final.

Ele espera encontrar arquivos `*_metrics.csv` em `reports/metricas/`, gerados pelos notebooks de treino.

## Criterio de Escolha

A escolha final considera:

1. F1-score, para equilibrar precision e recall.
2. Recall, porque falso negativo em saude e especialmente critico.
3. AUC-ROC, para avaliar separabilidade geral.
4. Tempo medio de inferencia por imagem.

O score de selecao usa ranking ponderado:

- 40% F1
- 30% recall
- 20% AUC-ROC
- 10% tempo de inferencia

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.training.comparison import (
    add_selection_scores,
    choose_final_model,
    export_final_checkpoint,
    load_metric_files,
)

config.ensure_project_directories()

if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

print("METRICS_DIR:", config.METRICS_DIR)
print("FIGURES_DIR:", config.FIGURES_DIR)
print("EXPORTED_MODELS_DIR:", config.EXPORTED_MODELS_DIR)

## Consolidacao das Metricas

Carregamos todos os arquivos individuais de metricas. Se a tabela estiver vazia, execute primeiro os notebooks de treino.

In [ ]:
all_metrics = load_metric_files(config.METRICS_DIR)

if all_metrics.empty:
    print("Nenhuma metrica encontrada. Execute os notebooks de treino antes desta comparacao.")
else:
    all_metrics.to_csv(config.METRICS_DIR / "model_comparison_all_metrics.csv", index=False)

all_metrics

## Ranking dos Modelos

O ranking abaixo combina desempenho e eficiencia. Quanto menor o `selection_score`, melhor a posicao do modelo.

In [ ]:
ranked_metrics = add_selection_scores(all_metrics)

if ranked_metrics.empty:
    print("Ranking indisponivel porque nao ha metricas consolidadas.")
else:
    ranked_metrics.to_csv(config.METRICS_DIR / "model_comparison_ranked.csv", index=False)

columns_to_show = [
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auc_roc",
    "seconds_per_image",
    "total_parameters",
    "trainable_parameters",
    "checkpoint_size_bytes",
    "selection_score",
    "source_file",
]
available_columns = [column for column in columns_to_show if column in ranked_metrics.columns]
ranked_metrics[available_columns] if not ranked_metrics.empty else ranked_metrics

## Graficos Comparativos

Geramos graficos para F1, AUC, tempo de inferencia, parametros e trade-off F1 versus inferencia.

In [ ]:
def save_bar_chart(data: pd.DataFrame, metric: str, title: str, output_name: str) -> None:
    if data.empty or metric not in data.columns:
        print(f"Grafico ignorado: {metric} indisponivel.")
        return

    plot_data = data.dropna(subset=[metric]).sort_values(metric, ascending=False)
    if plot_data.empty:
        print(f"Grafico ignorado: {metric} sem valores validos.")
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    if HAS_SEABORN:
        sns.barplot(data=plot_data, x="model_name", y=metric, ax=ax, color="#4C78A8")
    else:
        ax.bar(plot_data["model_name"], plot_data[metric], color="#4C78A8")
    ax.set_title(title)
    ax.set_xlabel("Modelo")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    fig.savefig(config.FIGURES_DIR / output_name, dpi=160, bbox_inches="tight")
    plt.show()


save_bar_chart(ranked_metrics, "f1", "F1-score por modelo", "comparacao_f1_por_modelo.png")
save_bar_chart(ranked_metrics, "auc_roc", "AUC-ROC por modelo", "comparacao_auc_por_modelo.png")
save_bar_chart(ranked_metrics, "seconds_per_image", "Tempo medio de inferencia por imagem", "comparacao_inferencia_por_modelo.png")
save_bar_chart(ranked_metrics, "total_parameters", "Total de parametros por modelo", "comparacao_parametros_por_modelo.png")

In [ ]:
if ranked_metrics.empty or {"seconds_per_image", "f1"} - set(ranked_metrics.columns):
    print("Grafico de trade-off indisponivel.")
else:
    plot_data = ranked_metrics.dropna(subset=["seconds_per_image", "f1"])
    if plot_data.empty:
        print("Grafico de trade-off sem valores validos.")
    else:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(plot_data["seconds_per_image"], plot_data["f1"], s=80, color="#59A14F")
        for _, row in plot_data.iterrows():
            ax.annotate(row["model_name"], (row["seconds_per_image"], row["f1"]), fontsize=8)
        ax.set_title("Trade-off F1-score versus tempo de inferencia")
        ax.set_xlabel("Segundos por imagem")
        ax.set_ylabel("F1-score")
        plt.tight_layout()
        fig.savefig(config.FIGURES_DIR / "comparacao_f1_vs_inferencia.png", dpi=160, bbox_inches="tight")
        plt.show()

## Escolha do Modelo Final

Selecionamos o melhor modelo pelo score ponderado. Depois copiamos o checkpoint escolhido para `models/exported/`, se o caminho do checkpoint existir.

In [ ]:
selected_model = choose_final_model(all_metrics)

if selected_model is None:
    print("Modelo final ainda nao pode ser escolhido porque nao ha metricas disponiveis.")
else:
    exported_checkpoint = export_final_checkpoint(selected_model, config.EXPORTED_MODELS_DIR)
    selected_model["exported_checkpoint_path"] = str(exported_checkpoint) if exported_checkpoint else None
    final_selection_df = pd.DataFrame([selected_model])
    final_selection_df.to_csv(config.METRICS_DIR / "final_model_selection.csv", index=False)
    display(final_selection_df.T)
    if exported_checkpoint:
        print("Checkpoint final exportado para:", exported_checkpoint)
    else:
        print("Checkpoint nao exportado. Verifique se checkpoint_path existe nas metricas.")

## Justificativa para o Relatorio

Use o texto abaixo como base para o relatorio tecnico depois de executar os treinos reais.

In [ ]:
if selected_model is None:
    print("Ainda nao ha modelo final. Execute os notebooks de treinamento para gerar metricas reais.")
else:
    justification = f"""
O modelo final selecionado foi {selected_model['model_name']}.
A escolha foi baseada em um ranking ponderado que prioriza F1-score e recall,
pois em um contexto de saude falso negativo e particularmente critico.
O modelo apresentou F1={selected_model.get('f1')}, recall={selected_model.get('recall')},
AUC-ROC={selected_model.get('auc_roc')} e tempo medio de inferencia de
{selected_model.get('seconds_per_image')} segundos por imagem.
Mesmo assim, o resultado deve ser interpretado como prototipo academico,
nao como ferramenta diagnostica real.
""".strip()
    print(justification)
    (config.REPORTS_DIR / "justificativa_modelo_final.txt").write_text(justification + "\n")

## Proxima Etapa

Depois de escolher o modelo final, seguir para as tarefas 23 e 24: prototipo em notebook e prototipo Flask.